# customer churn analysis

in this file we are going to connect the sql database and perform some analysis on the customer churn data. We will use Python libraries such as pandas, numpy, and matplotlib for data manipulation and visualization. The goal is to identify patterns and factors that contribute to customer churn, which can help in developing strategies to retain customers.

In [1]:
#import libaries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


# data source

In [ ]:
# import database
conn = sqlite3.connect('customer_churn.db')  #connection BUILDING

sql_query = """ SELECT name
                FROM sqlite_master
                WHERE type='table'
            """

tables = pd.read_sql(sql_query, conn)  
print("tables", tables)

print("============================================================")

# create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    globals()[f"df_{table_name}"] = df   #globals() function to create a variable with the name of the table
    print(f"Created dataframe: df_{table_name}")
    
# connection close
conn.close()

tables               name
0      db_customer
1  db_subscription
2       db_support
Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [11]:
#print table names and columns name
conn = sqlite3.connect('customer_churn.db')  #connection BUILDING

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column names
    columns_query = f"PRAGMA table_info({table_name});" #pargma a command to get the column names of a table
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist()) # print column names as list

    # connection close
conn.close()


Table Name: db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


# data cleaning

In [14]:
df_db_customer.head(5)

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [16]:
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,NaN,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,NaN,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,NaN,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,NaN,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,NaN,None


In [15]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


1. Total entries: 21
2. country has only 18 entries , so there is null 
3. intrests 4 , so drop this 
4. rename col " name" ==> customer_name
5. change data type of "date_of_birth" to datetime
6. fix gender column to have only 2 values male and female


In [21]:
# rename
df_db_customer.rename(columns ={'name':'customer_name'}, inplace = True) #inplace= true updates df
df_db_customer.head(5)

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [ ]:
#  drop columns ( intrest and pincode)

